# Data Preparation: `df_artists_final.csv`

This notebook is the **main data-preparation pipeline** for the Hitmakers project.  
It takes the upstream `df_artists.csv` and produces the model-ready `df_artists_final.csv`.

| | |
|---|---|
| **Output** | `df_artists_final.csv` — 759 rows × 26 columns |
| **Target** | `top_20_hitmaker` — 1 if artist had **> 1** top-20 song, 0 if exactly 1 |
| **Supplementary notebooks** | All upstream pipeline notebooks are collected in **`pipeline_supplement/`** |

---

## Upstream pipeline overview

The input `df_artists.csv` (13,655 artists × 44 columns) was built across **8 feature-engineering stages** plus a Google Trends experiment.  
The table below summarises each stage; detailed notebooks are in `pipeline_supplement/`.

### Raw data sources

| File | Description | Source |
|------|-------------|--------|
| `billboard_hot100_1958_2026.csv` | Week-by-week Billboard Hot 100 chart entries (1958–2026), ~350 K rows | External |
| `billboard_hot100_songs_final.csv` | Deduplicated Hot 100 songs (one row per unique song, ~30 K rows) | Derived from weekly data |
| `billboard_200_albums_final.csv` | Deduplicated Billboard 200 albums (one row per unique album, ~19 K rows) | Derived from weekly data |
| `kang_data_w_spotify.csv` | Kang/Kwon verified artist dataset with Spotify IDs | External academic dataset |
| `gabminamedez_spotify_data.csv` | Spotify audio features for songs | [GitHub: gabminamedez/spotify-data](https://github.com/gabminamedez/spotify-data) |
| `google_trends_top3000.csv` | Monthly Google Trends interest (YouTube + web search) for top artists | Pulled via `pytrends` API |
| MusicBrainz PostgreSQL DB (Docker) | Artist IDs, genre tags, collaboration edges | Local MusicBrainz database dump |

### Feature engineering stages

| Stage | Name | Notebook(s) in `pipeline_supplement/` | Key Transformations | Key Output |
|:-----:|------|---------------------------------------|---------------------|------------|
| 1 | **Billboard Cleaning & Artist Verification** | `billboard_data_cleaning_pt_1` | Split collaborations (e.g. "Drake Feat. Rihanna" → separate rows); verified names against MusicBrainz (fuzzy + alias matching); normalized `performer_normalized` | `billboard_hot100_songs_cleaned.csv`, `billboard_200_albums_cleaned.csv` |
| 2 | **Artist / Song Aggregation & Target** | `billboard_data_cleaning_pt_2`, `EDA_1_+billboard_data_cleaning_pt_3` | Aggregated per-artist chart stats (`total_charting_songs`, `#1_hit_count`, `top_10/20/50_counts`, …); computed milestone-year columns; created `top_10_hitmaker_songs` flag | `df_artists_basic.csv` → `df_artists_with_network_metrics.csv` |
| 3 | **Artist Name Deduplication** | `df_artists_clean` | Removed collab artifacts (`feat.`/`ft.`/`featuring`); deduplicated `the`/non-`the` variants; manual keep-lists for legit names (e.g. "Earth, Wind & Fire", "AC/DC") | ~13,600 → ~13,400 unique artists |
| 4 | **Label Aggregation & Genre Tagging** | `billboard_data_cleaning_pt_4`, `billboard_data_cleaning_pt_5_genre` | Aggregated record labels; queried MusicBrainz for genre tags; built **546-genre → 18-major-category** mapping; created `combined_major_genre_categories` | 18 one-hot genre columns per artist |
| 5 | **MusicBrainz ID Corrections** | `billboard_data_cleaning_pt_7_filling_in_missing_artist_ids` | Fixed 564 wrong MusicBrainz IDs; filled missing IDs from Kang dataset; assigned `musicbrainz_mbid` UUIDs | Corrected ID columns across all dataframes |
| 6 | **Song-Level Feature Engineering** | `df_songs_create` | Created `df_songs` with recording MBIDs, song/album genre tags, Spotify audio features (~30–40 % coverage), record label data | `df_songs.csv` (~30 K rows × ~37 cols) |
| 7 | **Network Metrics** | `billboard_data_cleaning_pt_8_new_network_metrics`, `billboard_network_data_merge` | Built collaboration network from MusicBrainz; computed graph metrics (degree, closeness, harmonic closeness, betweenness, eigenvector centrality); rolling 5-year windows 1958–2024 | `df_artists_network_metrics.csv` (26 cols) |
| 8 | **Column Cleanup & Final Assembly** | `billboard_data_cleaning_pt_6_condensing`, `df_artists_clean` (final cells) | Dropped redundant columns; merged Spotify features for first top-20 hit; created `top_20_hitmaker` target; added first-top-20-hit genre mapping | `df_artists.csv` (13,655 × 44) |
| — | **Google Trends** *(not in final model)* | `Googletrend_dataset`, `google_trends_engineered` (in `GS/`) | Pulled monthly Google Trends interest 2004–2021 for top 500 artists; engineered `google_trend_decay` feature | `df_songs_google_decay.csv` |

### 18 major genre categories

> Blues · Classical · Country/Americana · Easy Listening/Vocal · Electronic/Dance · Experimental/Avant-Garde · Folk · Gospel/Christian/Religious · Hip Hop/Rap · Jazz · Latin · Metal · Pop · Punk/Hardcore · R&B/Soul/Funk · Reggae/Caribbean · Rock · World Music

### Network metrics retained in the final model

| Metric | Description |
|--------|-------------|
| `harmonic_closeness_centrality` | Average distance to all other artists (handles disconnected components) |
| `betweenness_centrality` | How often an artist is a "bridge" between communities |
| `eigenvector_centrality` | Connected to other well-connected artists |

> All metrics are computed at the year of the artist's first top-10 hit, using a rolling 5-year collaboration window.

---

## This notebook's pipeline (`df_artists.csv` → `df_artists_final.csv`)

| Step | Operation | Details |
|:----:|-----------|---------|
| 1 | **Filter to 2000–2019** | Keep only artists whose first top-20 hit was between 2000 and 2019 |
| 2 | **Drop identifier / non-feature columns** | Remove MusicBrainz IDs, Spotify IDs, raw genre strings, etc. |
| 3 | **One-hot encode genre tags** | Expand `combined_major_genre_categories` into 18 binary genre columns + `unknown` flag + genre count |
| 4 | **Drop nulls & duplicates** | Remove rows with null target (`top_20_hitmaker`) and any duplicate rows |
| 5 | **Drop Spotify audio features** | ~40 % missing — too many nulls for reliable modelling |
| 6 | **Drop collinear network metrics** | Remove `degree_centrality` and `power_of_connected_artists` (correlated with eigenvector) |
| 7 | **Fill network metric nulls** | Fill remaining nulls with 0 (no collaboration data = no centrality) |

## 1 — Load source data

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

# --- Paths  ---
DATA_DIR = Path('Datasets/Main_Data')
INPUT_PATH = DATA_DIR / 'df_artists.csv'
OUTPUT_PATH = Path('.') / 'df_artists_final.csv'

df_artists = pd.read_csv(INPUT_PATH)
print(f"Loaded df_artists: {df_artists.shape}")
df_artists.head(2)

Loaded df_artists: (13655, 44)


,name,musicbrainz_artist_id,musicbrainz_mbid,spotify_id,performer_pre_normalized,first_top_20_hit_year,first_charting_song_year,last_charting_song_year,years_through_first_top_20_hit,#_of_charting_songs_through_first_top_20_hit,first_song_year,years_active_on_charts,first_charting_song_position,first_charting_song_duration,top_20_hit_song_#_wks_on_chart_any_position,genre_tags_musicbrainz,first_year_on_chart_songs,genre_tags_through_first_top_10_hit,major_genre_categories_through_first_top_10_hit,#_of_major_genre_categories_through_first_top_10_hit,musicbrainz_major_genre_categories,musicbrainz_#_of_genres,spotify_genres,spotify_major_genre_categories,combined_major_genre_categories,first_top_20_song_major_genres,first_top_20_song_duration_ms,first_top_20_song_acousticness,first_top_20_song_danceability,first_top_20_song_energy,first_top_20_song_instrumentalness,first_top_20_song_liveness,first_top_20_song_loudness,first_top_20_song_speechiness,first_top_20_song_tempo,first_top_20_song_valence,first_top_20_song_mode,first_top_20_song_explicit,degree_centrality_top20_rolling5,harmonic_closeness_centrality_top20_rolling5,betweenness_centrality_top20_rolling5,eigenvector_centrality_top20_rolling5,power_of_connected_artists_top20_rolling5,top_20_hitmaker
0,!!! (chk chk chk),NaN,NaN,NaN,!!! (Chk Chk Chk),NaN,NaN,NaN,NaN,NaN,NaN,2007-2007,0,0,NaN,NaN,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"""groove"" holmes",NaN,NaN,NaN,"""Groove"" Holmes",NaN,1966.0,1966.0,NaN,NaN,1966.0,1966-1966,44,11,NaN,NaN,1966.0,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2 — Filter to 2000–2019 & drop non-feature columns

In [8]:
# Columns that are identifiers, metadata, or redundant with other features
COLS_TO_DROP = [
    # Identifiers / metadata
    'musicbrainz_artist_id', 'musicbrainz_mbid', 'spotify_id',
    'performer_pre_normalized',
    # Time-range columns not used as features
    'last_charting_song_year', 'first_song_year', 'years_active_on_charts',
    'first_charting_song_position', 'first_charting_song_duration',
    # Raw genre tag strings (we one-hot encode combined_major_genre_categories instead)
    'genre_tags_musicbrainz', 'first_year_on_chart_songs',
    'genre_tags_through_first_top_10_hit',
    'major_genre_categories_through_first_top_10_hit',
    '#_of_major_genre_categories_through_first_top_10_hit',
    'musicbrainz_major_genre_categories', 'musicbrainz_#_of_genres',
    'spotify_genres', 'spotify_major_genre_categories',
]

df = (
    df_artists[df_artists['first_top_20_hit_year'].between(2000, 2019)]
    .drop(columns=[c for c in COLS_TO_DROP if c in df_artists.columns])
    .rename(columns={'combined_major_genre_categories': 'combined_major_genre_categories_artist'})
    .reset_index(drop=True)
)

print(f"After filter + drop: {df.shape}")
print(f"Year range: {df['first_top_20_hit_year'].min()} – {df['first_top_20_hit_year'].max()}")

After filter + drop: (778, 26)
Year range: 2000.0 – 2019.0


## 3 — One-hot encode artist genre

In [9]:
# Count genres per artist (before one-hot encoding)
genre_counts = (
    df['combined_major_genre_categories_artist']
    .fillna('')
    .apply(lambda x: len([g for g in x.split(', ') if g.strip()]) if x else 0)
)

# Explode comma-separated genres into one-hot columns prefixed 'artist_genre_'
artist_genre_dummies = (
    df['combined_major_genre_categories_artist']
    .fillna('')
    .str.split(', ')
    .explode()
    .str.strip()
    .pipe(lambda s: pd.get_dummies(s, prefix='artist_genre'))
    .groupby(level=0)
    .max()
)

# Remove empty-string column (artifact from NaN rows)
artist_genre_dummies = artist_genre_dummies.loc[
    :, artist_genre_dummies.columns != 'artist_genre_'
]

# Insert genre dummies where the original column was
insert_at = df.columns.get_loc('combined_major_genre_categories_artist')
df = pd.concat(
    [df.iloc[:, :insert_at], artist_genre_dummies, df.iloc[:, insert_at + 1:]],
    axis=1,
)

# Add 'artist_genre_unknown' flag for artists with zero genre tags
all_genre_cols = [c for c in df.columns if c.startswith('artist_genre_')]
last_genre_idx = df.columns.get_loc(all_genre_cols[-1])
df.insert(
    last_genre_idx + 1,
    'artist_genre_unknown',
    (df[all_genre_cols].sum(axis=1) == 0).astype(int),
)

# Add genre count column right after 'artist_genre_unknown'
unknown_idx = df.columns.get_loc('artist_genre_unknown')
df.insert(unknown_idx + 1, '#_of_genres_artist', genre_counts)

print(f"Genre columns: {artist_genre_dummies.columns.tolist()}")
print(f"Shape after genre encoding: {df.shape}")

Genre columns: ['artist_genre_Blues', 'artist_genre_Classical', 'artist_genre_Country/Americana', 'artist_genre_Easy Listening/Vocal', 'artist_genre_Electronic/Dance', 'artist_genre_Experimental/Avant-Garde', 'artist_genre_Folk', 'artist_genre_Gospel/Christian/Religious', 'artist_genre_Hip Hop/Rap', 'artist_genre_Jazz', 'artist_genre_Latin', 'artist_genre_Metal', 'artist_genre_Pop', 'artist_genre_Punk/Hardcore', 'artist_genre_R&B/Soul/Funk', 'artist_genre_Reggae/Caribbean', 'artist_genre_Rock', 'artist_genre_World Music']
Shape after genre encoding: (778, 45)


## 4 — Drop nulls, duplicates, identifier columns

In [10]:
# Drop rows where target is null (artists without a verified top-20 song)
df = (
    df
    .dropna(subset=['top_20_hitmaker'])
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"After dropping nulls & dupes: {df.shape}")
print(f"\nTarget distribution:\n{df['top_20_hitmaker'].value_counts()}")

After dropping nulls & dupes: (759, 45)

Target distribution:
top_20_hitmaker
0.0    431
1.0    328
Name: count, dtype: int64


In [11]:
# Drop 'name' (identifier, not a feature) and cast bool genre columns to int
df = df.drop(columns=['name'])

genre_cols = [c for c in df.columns if c.startswith('genre_')]
df[genre_cols] = df[genre_cols].astype(int)

print(f"Shape after dropping name: {df.shape}")

Shape after dropping name: (759, 44)


## 5 — Handle missing values & drop low-signal features

In [12]:
# --- Drop Spotify audio features (too many nulls, ~40% missing) ---
SPOTIFY_AUDIO_COLS = [
    'first_top_20_song_duration_ms', 'first_top_20_song_acousticness',
    'first_top_20_song_danceability', 'first_top_20_song_energy',
    'first_top_20_song_instrumentalness', 'first_top_20_song_liveness',
    'first_top_20_song_loudness', 'first_top_20_song_speechiness',
    'first_top_20_song_tempo', 'first_top_20_song_valence',
    'first_top_20_song_mode', 'first_top_20_song_key',
    'first_top_20_song_explicit', 'first_top_20_song_popularity',
]
df = df.drop(columns=[c for c in SPOTIFY_AUDIO_COLS if c in df.columns])

# --- Drop collinear network metrics ---
# degree_centrality and power_of_connected_artists are highly correlated
# with eigenvector_centrality; keeping only the 3 least-correlated metrics.
df = df.drop(columns=[
    'degree_centrality_top20_rolling5',
    'power_of_connected_artists_top20_rolling5',
], errors='ignore')

# --- Fill remaining network metric nulls with 0 ---
# 0 = no collaboration data / no centrality in the co-charting network
NETWORK_COLS = [
    'harmonic_closeness_centrality_top20_rolling5',
    'betweenness_centrality_top20_rolling5',
    'eigenvector_centrality_top20_rolling5',
]
df[NETWORK_COLS] = df[NETWORK_COLS].fillna(0)

# --- Drop remaining non-feature columns ---
df = df.drop(columns=[
    'first_top_20_song_major_genres',
    'first_charting_song_year',
    'first_top_20_hit_year',
], errors='ignore')

print(f"Final shape: {df.shape}")
print(f"\nRemaining nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("✓ No missing values")

Final shape: (759, 27)

Remaining nulls:
top_20_hit_song_#_wks_on_chart_any_position    87
dtype: int64


## 6 — Final validation & save

In [13]:
# Quick sanity checks
print("=== Final Dataset Summary ===")
print(f"Shape: {df.shape}")

# --- Readable column listing grouped by category ---
print(f"\nColumns ({len(df.columns)}):\n")

col_groups = {
    "Target":          [c for c in df.columns if c == 'top_20_hitmaker'],
    "Chart stats":     [c for c in df.columns if any(k in c for k in
                        ['charting_songs', 'charting_albums', 'hit_count',
                         'top_10', 'top_20', 'top_50', 'highest_charting',
                         'wks_on_chart', 'num1'])
                        and 'genre' not in c and 'centrality' not in c],
    "Genre (one-hot)": [c for c in df.columns if c.startswith('artist_genre_')
                        or c == '#_of_genres_artist'],
    "Network metrics": [c for c in df.columns if 'centrality' in c],
}
col_groups["Other"] = [c for c in df.columns
                       if not any(c in grp for grp in col_groups.values())]

for group, cols in col_groups.items():
    if cols:
        print(f"  {group} ({len(cols)}):")
        for c in cols:
            print(f"    • {c}")
        print()

print(f"Target distribution:\n{df['top_20_hitmaker'].value_counts()}")
print(f"\nDtypes:\n{df.dtypes.value_counts()}")
print(f"\nNull count: {df.isnull().sum().sum()}")

df.head()

=== Final Dataset Summary ===
Shape: (759, 27)

Columns (27):

  Target (1):
    • top_20_hitmaker

  Chart stats (4):
    • years_through_first_top_20_hit
    • #_of_charting_songs_through_first_top_20_hit
    • top_20_hit_song_#_wks_on_chart_any_position
    • top_20_hitmaker

  Genre (one-hot) (20):
    • artist_genre_Blues
    • artist_genre_Classical
    • artist_genre_Country/Americana
    • artist_genre_Easy Listening/Vocal
    • artist_genre_Electronic/Dance
    • artist_genre_Experimental/Avant-Garde
    • artist_genre_Folk
    • artist_genre_Gospel/Christian/Religious
    • artist_genre_Hip Hop/Rap
    • artist_genre_Jazz
    • artist_genre_Latin
    • artist_genre_Metal
    • artist_genre_Pop
    • artist_genre_Punk/Hardcore
    • artist_genre_R&B/Soul/Funk
    • artist_genre_Reggae/Caribbean
    • artist_genre_Rock
    • artist_genre_World Music
    • artist_genre_unknown
    • #_of_genres_artist

  Network metrics (3):
    • harmonic_closeness_centrality_top20_rolling5
   

,years_through_first_top_20_hit,#_of_charting_songs_through_first_top_20_hit,top_20_hit_song_#_wks_on_chart_any_position,artist_genre_Blues,artist_genre_Classical,artist_genre_Country/Americana,artist_genre_Easy Listening/Vocal,artist_genre_Electronic/Dance,artist_genre_Experimental/Avant-Garde,artist_genre_Folk,artist_genre_Gospel/Christian/Religious,artist_genre_Hip Hop/Rap,artist_genre_Jazz,artist_genre_Latin,artist_genre_Metal,artist_genre_Pop,artist_genre_Punk/Hardcore,artist_genre_R&B/Soul/Funk,artist_genre_Reggae/Caribbean,artist_genre_Rock,artist_genre_World Music,artist_genre_unknown,#_of_genres_artist,harmonic_closeness_centrality_top20_rolling5,betweenness_centrality_top20_rolling5,eigenvector_centrality_top20_rolling5,top_20_hitmaker
0,1.0,9.0,27.0,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,0,1,5314.363492,0.010130,1.510211e-01,1.0
1,2.0,15.0,41.0,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,0,1,5233.436039,0.001326,7.977090e-02,1.0
2,1.0,2.0,53.0,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,True,False,0,3,0.000000,0.000000,0.000000e+00,1.0
3,1.0,1.0,37.0,False,False,False,False,True,False,False,False,True,False,False,False,True,True,False,False,True,False,0,5,0.000000,0.000000,0.000000e+00,1.0
4,1.0,6.0,20.0,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,False,0,2,1.000000,0.000000,1.555431e-18,1.0


In [ ]:
# # Save to CSV
# df.to_csv(OUTPUT_PATH, index=False)
# print(f"✓ Saved df_artists_final.csv ({df.shape[0]} rows × {df.shape[1]} cols)")

✓ Saved df_artists_final.csv (759 rows × 27 cols)
